<a href="https://colab.research.google.com/github/Nidaa777-3/yolo_traffic/blob/main/yolov8n_traffic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ultralytics
import os
import shutil
import pandas as pd
from tqdm import tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.3 MB/s eta 0:00:00


In [2]:
!unzip -q archive.zip -d /content/raw_data

In [3]:
base_dir = '/content/yolo_dataset'
for split in ['train', 'val', 'test']:
    os.makedirs(f'{base_dir}/images/{split}', exist_ok=True)
    os.makedirs(f'{base_dir}/labels/{split}', exist_ok=True)

def process_folder(source_path, target_split):
    print(f"Processing {source_path}...")
    for class_id in tqdm(os.listdir(source_path)):
        class_dir = os.path.join(source_path, class_id)
        if not os.path.isdir(class_dir): continue

        images = [f for f in os.listdir(class_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        for i, img_name in enumerate(images):
            # For the 'data' folder, we split 80% to train and 20% to val
            if target_split == 'train_val':
                current_split = 'val' if i % 5 == 0 else 'train'
            else:
                current_split = 'test'

            # Move Image
            src_img = os.path.join(class_dir, img_name)
            dst_img = f'{base_dir}/images/{current_split}/{class_id}_{img_name}'
            shutil.copy(src_img, dst_img)

            # Create Label (Full-frame detection: class x_center y_center width height)
            label_path = f'{base_dir}/labels/{current_split}/{class_id}_{img_name.rsplit(".", 1)[0]}.txt'
            with open(label_path, 'w') as f:
                # YOLO format: ClassID 0.5 0.5 1.0 1.0 (The sign is the whole image)
                f.write(f"{class_id} 0.5 0.5 1.0 1.0")

# Run the processing
# Assuming your unzipped folders are at /content/raw_data/data and /content/raw_data/test
process_folder('/content/raw_data/DATA', 'train_val')
process_folder('/content/raw_data/TEST', 'test')

print("\nDone! Data is now in YOLO format.")

Processing /content/raw_data/DATA...


100%|██████████| 52/52 [00:01<00:00, 51.64it/s]


Processing /content/raw_data/TEST...


100%|██████████| 52/52 [00:00<00:00, 663.14it/s]


Done! Data is now in YOLO format.


In [4]:
yaml_content = f"""
path: /content/yolo_dataset
train: images/train
val: images/val

names:
  0: Speed limit (5km/h)
  1: Speed limit (15km/h)
  2: Speed limit (30km/h)
  3: Speed limit (40km/h)
  4: Speed limit (50km/h)
  5: Speed limit (60km/h)
  6: Speed limit (70km/h)
  7: speed limit (80km/h)
  8: Dont Go straight or left
  9: Unknown7
  10: Dont Go straight
  11: Dont Go Left
  12: Dont Go Left or Right
  13: Dont Go Right
  14: Dont overtake from Left
  15: No Uturn
  16: No Car
  17: No horn
  18: No entry
  19: No stopping
  20: Go straight or right
  21: Go straight
  22: Go Left
  23: Go Left or right
  24: Go Right
  25: keep Left
  26: keep Right
  27: Roundabout mandatory
  28: watch out for cars
  29: Horn
  30: Bicycles crossing
  31: Uturn
  32: Road Divider
  33: Unknown6
  34: Danger Ahead
  35: Zebra Crossing
  36: Bicycles crossing
  37: Children crossing
  38: Dangerous curve to the left
  39: Dangerous curve to the right
  40: Unknown1
  41: Unknown2
  42: Unknown3
  43: Go right or straight
  44: Go left or straight
  45: Unknown4
  46: ZigZag Curve
  47: Train Crossing
  48: Under Construction
  49: Unknown5
  50: Fences
  51: Heavy Vehicle Accidents
"""

with open('/content/yolo_dataset/data.yaml', 'w') as f:
    f.write(yaml_content)

In [5]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model.train(data='/content/yolo_dataset/data.yaml', epochs=30, imgsz=640)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, 

In [7]:
# Run validation on the test images
metrics = model.val()
print(f"Model Accuracy (mAP50): {metrics.box.map50}")

# Try a random prediction from your test folder
import glob
import random
test_images = glob.glob('/content/raw_data/TEST/*/*.png')
sample = random.choice(test_images)

results = model.predict(source=sample, save=True, conf=0.5)

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1392.2±884.8 MB/s, size: 33.1 KB)
val: Scanning /content/yolo_dataset/labels/val.cache... 1156 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1156/1156 323.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 73/73 5.9it/s 12.4s
                   all       1156       1156       0.97      0.971      0.988      0.988
   Speed limit (5km/h)         26         26      0.991          1      0.995      0.995
  Speed limit (15km/h)         10         10       0.98          1      0.995      0.995
  Speed limit (30km/h)         21         21      0.989          1      0.995      0.995
  Speed limit (40km/h)         63         63      0.996          1      0.995      0.995
  Speed limit (50km/h)         29         29      0.992          1      0.995      0.995
  Speed limit (60km/h)         4